In [1]:
import os
os.chdir("/Users/nautilus/cobblestone_study")

import sys
sys.path.insert(0, "/Users/nautilus/cobblestone_study")

import pandas as pd
from config import CLEAN_PARQUET
from src.curve import build_signal

df = pd.read_parquet("/Users/nautilus/cobblestone_study/data/clean.parquet")
signal = build_signal(df)
signal


{'period_start': '2026-06-05 00:00:00+02:00',
 'period_end': '2026-06-11 23:00:00+02:00',
 'model_mae': 16.66,
 'model_rmse': 28.58,
 'baseload': {'forecast': np.float64(93.55),
  'reference': np.float64(100.41),
  'spread': np.float64(-6.86),
  'signal': 'SHORT',
  'conviction': 'LOW',
  'zscore': np.float64(-0.24)},
 'peak': {'forecast': np.float64(76.01),
  'reference': np.float64(78.59),
  'spread': np.float64(-2.58),
  'signal': 'SHORT',
  'conviction': 'LOW',
  'zscore': np.float64(-0.09)}}

In [10]:
prompt_window = df.iloc[-(7 * 24):]

# Driver 1. avg residual load 
avg_residual = round(prompt_window["residual_load_fc"].mean(), 0)

spike_hours = prompt_window["prices"].groupby(prompt_window.index.hour).mean().nlargest(3)

print(f"Avg residual load: {avg_residual} MW")
print("Top spike hours:")

spike_hours_fmt = [f"{h:02d}:00 (~{p:.0f} €/MWh)" for h, p in spike_hours.items()]
print(spike_hours_fmt)


Avg residual load: 28346.0 MW
Top spike hours:
['20:00 (~173 €/MWh)', '21:00 (~170 €/MWh)', '22:00 (~150 €/MWh)']


In [12]:
s = signal
b, p = s["baseload"], s["peak"]

prompt = f"""You are a power trading analyst. Write a concise daily fair-value note for the
German (DE_LU) day-ahead market, for the front-week prompt period {s['period_start'][:10]} to {s['period_end'][:10]}.

DATA (do not invent numbers beyond these):
- Baseload: forecast {b['forecast']} €/MWh vs reference {b['reference']} €/MWh (spread {b['spread']:+}), signal {b['signal']}, conviction {b['conviction']}.
- Peak: forecast {p['forecast']} €/MWh vs reference {p['reference']} €/MWh (spread {p['spread']:+}), signal {p['signal']}, conviction {p['conviction']}.
- Model out-of-sample MAE: {s['model_mae']} €/MWh (edge below this is noise).
- Avg residual load over the week: {avg_residual:.0f} MW.
- Highest-price hours: {", ".join(spike_hours_fmt)}.

Reference is a trailing-4-week realized-price proxy for the forward curve.

Respond with ONLY valid JSON (no markdown, no backticks) using exactly these keys:
{{
  "headline": "one-sentence summary of the view",
  "baseload_view": "1-2 sentences: direction, conviction, why",
  "peak_view": "1-2 sentences: direction, conviction, why",
  "key_drivers": ["short bullet", "short bullet"],
  "risks": ["what would invalidate this view", "..."]
}}"""

print(prompt)


You are a power trading analyst. Write a concise daily fair-value note for the
German (DE_LU) day-ahead market, for the front-week prompt period 2026-06-05 to 2026-06-11.

DATA (do not invent numbers beyond these):
- Baseload: forecast 93.55 €/MWh vs reference 100.41 €/MWh (spread -6.86), signal SHORT, conviction LOW.
- Peak: forecast 76.01 €/MWh vs reference 78.59 €/MWh (spread -2.58), signal SHORT, conviction LOW.
- Model out-of-sample MAE: 16.66 €/MWh (edge below this is noise).
- Avg residual load over the week: 28346 MW.
- Highest-price hours: 20:00 (~173 €/MWh), 21:00 (~170 €/MWh), 22:00 (~150 €/MWh).

Reference is a trailing-4-week realized-price proxy for the forward curve.

Respond with ONLY valid JSON (no markdown, no backticks) using exactly these keys:
{
  "headline": "one-sentence summary of the view",
  "baseload_view": "1-2 sentences: direction, conviction, why",
  "peak_view": "1-2 sentences: direction, conviction, why",
  "key_drivers": ["short bullet", "short bullet"]

In [13]:
import os
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={"temperature": 0.2},
)

raw = response.text
print(raw)


{
  "headline": "DE_LU Day-Ahead Front-Week indicates a slight bearish bias for both Baseload and Peak, though forecasts remain within the model's noise threshold.",
  "baseload_view": "Our Baseload forecast of 93.55 €/MWh suggests a SHORT signal against the 100.41 €/MWh reference, but conviction is LOW as the -6.86 €/MWh spread is well within the model's 16.66 €/MWh MAE.",
  "peak_view": "The Peak forecast of 76.01 €/MWh also indicates a SHORT signal versus the 78.59 €/MWh reference, with LOW conviction due to the -2.58 €/MWh spread being within the model's 16.66 €/MWh MAE.",
  "key_drivers": [
    "Forecasted prices below reference for both Baseload and Peak.",
    "Average residual load of 28346 MW for the week.",
    "Evening price spikes (20:00-22:00) reaching up to ~173 €/MWh."
  ],
  "risks": [
    "Forecasted prices moving within the model's 16.66 €/MWh MAE, invalidating the current bearish bias.",
    "Unforeseen shifts in residual load or evening demand patterns."
  ]
}


In [14]:
import json

REQUIRED_FIELDS = ["headline", "baseload_view", "peak_view", "key_drivers", "risks"]

text = raw.strip()
# Strip code fences if the model added them
if text.startswith("```"):
    text = text.split("```")[1].lstrip("json").strip()

note = json.loads(text)   # raises if not valid JSON

missing = [f for f in REQUIRED_FIELDS if f not in note]
if missing:
    raise ValueError(f"Missing fields: {missing}")

print("Valid! All fields present.")
print(note["headline"])


Valid! All fields present.
DE_LU Day-Ahead Front-Week indicates a slight bearish bias for both Baseload and Peak, though forecasts remain within the model's noise threshold.


In [ ]:
from datetime import datetime, timezone

os.makedirs("outputs", exist_ok=True)

entry = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "gemini-2.5-flash",
    "prompt": prompt,
    "response": raw,
}

with open("outputs/llm_logs.jsonl", "a") as f:
    f.write(json.dumps(entry) + "\n")

print("Logged to outputs/llm_logs.jsonl")
